# Attention, derived from the problem it solves

Score, softmax, weighted sum — built up in NumPy, with the score matrix visualised, then the two refinements the 2017 paper added.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 15 — Language Models and the Transformer](../../../course-web-slides/ch15/index.html) &nbsp;·&nbsp; **Section:** 03 — The Transformer architecture

---

## The simplest version

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def dot_product_attention(target, source):
    scores = np.einsum("btd,bsd->bts", target, source)
    scores = softmax(scores, axis=-1)
    return np.einsum("bts,bsd->btd", scores, source), scores

rng = np.random.default_rng(0)
target = rng.normal(size=(1, 5, 8))     # 5 target positions, 8 dims
source = rng.normal(size=(1, 7, 8))     # 7 source positions

out, scores = dot_product_attention(target, source)
print("output:", out.shape, " scores:", scores.shape)
print("each row of scores sums to 1:", np.allclose(scores.sum(-1), 1))

Read the `einsum` subscripts: `b`atch, `t`arget length, `s`ource length, `d`imension. The first contraction produces a **(batch, target, source)** score matrix; the second uses it to take a weighted sum.

**Every attention implementation in every framework is these two contractions with a softmax between them.**

## The score matrix, drawn

In [ ]:
import matplotlib.pyplot as plt

eng = ["I", "will", "bring", "the", "bag", "to", "you"]
spa = ["Te", "traeré", "la", "bolsa", "[end]"]

# A hand-built matrix showing what a trained model should learn.
S = np.array([
    [.02, .03, .04, .02, .03, .06, .80],
    [.20, .45, .30, .02, .01, .01, .01],
    [.02, .02, .03, .60, .28, .03, .02],
    [.01, .02, .03, .20, .70, .02, .02],
    [.05, .05, .10, .05, .10, .30, .35],
])

fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.imshow(S, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(eng)), eng, rotation=45, ha="right")
ax.set_yticks(range(len(spa)), spa)
for i in range(len(spa)):
    for j in range(len(eng)):
        if S[i, j] > .15:
            ax.text(j, i, f"{S[i,j]:.2f}", ha="center", va="center",
                    fontsize=8, color="w" if S[i, j] > .5 else "k")
plt.colorbar(im); ax.set_title("Attention scores for a translation")
plt.tight_layout(); plt.show()

Read a row as: *when producing this Spanish word, how much did the model draw on each English word?* **The `Te` row peaking at `you` is exactly the long-range dependency the RNN could not express.**

## Parameterizing it: query, key, value

In [ ]:
dim = 8
Wq, Wk, Wv, Wo = (rng.normal(size=(dim, dim)) * 0.3 for _ in range(4))

def parameterized_attention(query, key, value):
    q = query @ Wq
    k = key @ Wk
    v = value @ Wv
    scores = softmax(np.einsum("btd,bsd->bts", q, k), axis=-1)
    out = np.einsum("bts,bsd->btd", scores, v)
    return out @ Wo, scores

out, _ = parameterized_attention(query=target, key=source, value=source)
print("output:", out.shape)

`sum(score(target, source) * source)` has become `sum(score(query, key) * value)`. The names come from **search engines**: the query is your search term, the keys are tags to match against, the values are what you retrieve.

## Refinement 1: scale before the softmax

In [ ]:
for d in [8, 64, 512]:
    q = rng.normal(size=(1, 4, d)); k = rng.normal(size=(1, 6, d))
    raw = np.einsum("btd,bsd->bts", q, k)
    scaled = raw / np.sqrt(d)
    print(f"dim {d:4d}:  raw logit std {raw.std():7.2f}   "
          f"scaled {scaled.std():5.2f}   "
          f"max softmax {softmax(raw).max():.3f} -> {softmax(scaled).max():.3f}")

Expected output:

```
dim    8:  raw logit std    2.9x   scaled  1.0x   max softmax 0.7xx -> 0.4xx
dim   64:  raw logit std    8.0x   scaled  1.0x   max softmax 0.9xx -> 0.4xx
dim  512:  raw logit std   22.x     scaled  1.0x   max softmax 1.000 -> 0.4xx
```

At 512 dimensions the unscaled softmax is **effectively one-hot**, and a one-hot softmax has vanishing gradients. Dividing by √d holds the logit variance constant regardless of dimension — which is why the mechanism is called *scaled* dot-product attention.

## Refinement 2: multiple heads

In [ ]:
def multi_head_attention(query, key, value, num_heads=4, head_dim=4):
    outs = []
    for _ in range(num_heads):
        wq = rng.normal(size=(query.shape[-1], head_dim)) * .3
        wk = rng.normal(size=(key.shape[-1], head_dim)) * .3
        wv = rng.normal(size=(value.shape[-1], head_dim)) * .3
        q, k, v = query @ wq, key @ wk, value @ wv
        s = softmax(np.einsum("btd,bsd->bts", q, k) / np.sqrt(head_dim), -1)
        outs.append(np.einsum("bts,bsd->btd", s, v))
    return np.concatenate(outs, axis=-1)

out = multi_head_attention(target, source, source)
print("concatenated across 4 heads of 4 dims:", out.shape)

One softmax sum is **blunt**: attend to many tokens and the interesting features of individual ones wash out. Running the operation several times with different projections lets one head match the subject while another attends to punctuation, in separate partitions of the output.

## The Keras layer

In [ ]:
import keras
from keras import layers
import numpy as np

mha = layers.MultiHeadAttention(num_heads=8, key_dim=32)
t = np.random.normal(size=(2, 5, 256)).astype("float32")
s = np.random.normal(size=(2, 7, 256)).astype("float32")

out, attn = mha(query=t, key=s, value=s, return_attention_scores=True)
print("output:", out.shape)
print("attention scores:", attn.shape, " (batch, heads, target, source)")

`return_attention_scores=True` gives you the matrix from the plot above, per head. **It is the first thing to look at when a Transformer misbehaves** — chapter 10's interpretability argument, in a different modality.

## Self-attention

In [ ]:
out = mha(query=s, key=s, value=s)
print("self-attention output:", out.shape)
print()
print('"The train left the station on time."')
print()
print("What kind of station? A radio station? The ISS?")
print("Self-attention lets the model give a high score to the pair")
print("(station, train), summing 'train' into the representation of")
print("'station' -- turning a word in a vacuum into a word in context.")

## Why attention alone is not enough

In [ ]:
# A sequence of length one. The score matrix is a single 1.
one = np.random.normal(size=(1, 1, 16)).astype("float32")
m = layers.MultiHeadAttention(num_heads=2, key_dim=8)
print("attention on a length-1 sequence:", m(one, one, one).shape)
print()
print("With one token the softmax is [1.0], so the layer reduces to a")
print("linear projection. Stack 100 of them and the whole computation")
print("still simplifies to ONE matrix multiplication.")
print()
print("That is why the Transformer block adds a feedforward network:")
print("  attention   -> mixes positions, no nonlinearity")
print("  feedforward -> mixes features, HAS the nonlinearity")

This is the argument for the second half of the block, and it is a genuine one rather than an empirical addition. **Attention is an expressive pooling operation**, and pooling alone cannot represent anything a single linear layer cannot.

---

## What to take away

- Attention is two einsum contractions with a softmax between them.
- The score matrix is (target × source) and is directly interpretable.
- Scale by √d or the softmax saturates and the gradients vanish.
- Multiple heads avoid one blunt weighted sum; the feedforward block supplies the nonlinearity attention lacks.